In [19]:
!pip install -q -r {os.path.join(torch.hub.get_dir(), "CompVis_DisMo_main", "requirements.txt")}
# transformers / opencv / sklearn are already on Kaggle; nothing extra needed for the VideoMAE side

In [20]:
!find /kaggle/input -name '*.mp4' | head
!ls /kaggle/input

/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/gv-CNrTbyr8.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/6UihMjZSmYY.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/C8Vwgdv_V0k.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/2XLhI47SOdo.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/owahEqTQQY0.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/RGGwnzGmWQY.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/FK7twgczIFc.mp4
/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train/drinking beer/Bz_UzA4A

In [22]:
"""
Appearance-vs-motion bias pipeline — unified across backbones.

One interface:  Backbone.forward(frames_uint8_rgb) -> {"emb": (D,), "logits": (C,) | None}
Two metrics, run identically per backbone:
  (1) perturbation sensitivity : how much each perturbation moves the representation
                                 (embedding cosine; + pred-KL / top1-kept if the model has a head)
  (2) linear probe             : freeze backbone, train a linear classifier on CLEAN embeddings,
                                 then test it on PERTURBED embeddings -> accuracy-under-perturbation

Shared, model-agnostic:  load_frames (cv2) and PERTURBATIONS.
Add a backbone (e.g. DINOv2 image baseline) by subclassing Backbone — nothing else changes.

Kaggle: GPU = T4 (P100 is sm_60 and incompatible with the stock torch). Internet ON.
        DisMo deps were installed via the repo's requirements.txt; HF_TOKEN if weights 401.
"""

import os, glob
import numpy as np
import torch
import cv2
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_FRAMES = 16          # VideoMAE-base needs 16; DisMo needs >=8 (uses N-4) -> 16 feeds both fairly

# ----------------------------------------------------------------------
# Shared: decode + uniformly sample NUM_FRAMES as (T, H, W, 3) uint8 RGB
# ----------------------------------------------------------------------
def load_frames(path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise RuntimeError(f"decoded 0 frames from {path}")
    frames = np.stack(frames)
    idx = np.linspace(0, len(frames) - 1, num_frames).astype(int)
    return frames[idx]

# ----------------------------------------------------------------------
# Shared perturbations: (T,H,W,3) uint8 -> same. Temporal vs appearance.
# ----------------------------------------------------------------------
def clean(f):     return f
def shuffle(f):   return f[np.random.permutation(len(f))]          # temporal: destroy order
def reverse(f):   return f[::-1].copy()                            # temporal: flip arrow of time
def freeze(f):    return np.repeat(f[len(f) // 2][None], len(f), 0)  # temporal: kill motion (also OOD)
def grayscale(f):                                                  # appearance: remove colour
    g = (f.astype(np.float32) @ np.array([0.299, 0.587, 0.114], np.float32))[..., None]
    return np.repeat(g, 3, axis=-1).astype(np.uint8)
def blur(f, k=15):                                                 # appearance: remove fine detail
    return np.stack([cv2.GaussianBlur(x, (k, k), 0) for x in f])

PERTURBATIONS = {"clean": clean, "shuffle": shuffle, "reverse": reverse,
                 "freeze": freeze, "grayscale": grayscale, "blur": blur}

# ----------------------------------------------------------------------
# Backbones
# ----------------------------------------------------------------------
class Backbone:
    name = "base"
    has_logits = False
    def forward(self, frames):           # -> {"emb": np.ndarray(D,), "logits": np.ndarray(C,) | None}
        raise NotImplementedError

class VideoMAEBackbone(Backbone):
    name = "videomae"
    has_logits = True
    def __init__(self, ckpt="MCG-NJU/videomae-base-finetuned-kinetics"):
        from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
        self.proc = VideoMAEImageProcessor.from_pretrained(ckpt)
        self.model = VideoMAEForVideoClassification.from_pretrained(ckpt).to(DEVICE).eval()
    @torch.no_grad()
    def forward(self, frames):
        inputs = self.proc(list(frames), return_tensors="pt").to(DEVICE)
        out = self.model(**inputs, output_hidden_states=True)
        emb = out.hidden_states[-1].squeeze(0).mean(0).float().cpu().numpy()   # (768,)
        logits = out.logits.squeeze(0).float().cpu().numpy()                   # (400,)
        return {"emb": emb, "logits": logits}

class DisMoBackbone(Backbone):
    name = "dismo"
    has_logits = False               # pure representation model: no classifier head
    def __init__(self):
        self.model = torch.hub.load("CompVis/DisMo", "motion_extractor_large",
                                    trust_repo=True).to(DEVICE).eval()
    @torch.no_grad()
    def forward(self, frames):
        f = np.stack([cv2.resize(x, (256, 256)) for x in frames])              # DisMo wants 256
        t = torch.from_numpy(f).float().div(255).mul(2).sub(1)                 # -> (-1, 1)
        t = t.unsqueeze(0).to(DEVICE)                                          # (1,T,256,256,3) THWC
        seq = self.model.forward_sliding(t).squeeze(0)                         # (T-4, D)  [verify shape once]
        emb = seq.reshape(-1, seq.shape[-1]).mean(0).float().cpu().numpy()     # mean-pool over time -> (D,)
        return {"emb": emb, "logits": None}

# To add a pure-appearance baseline later:
# class DINOv2Backbone(Backbone): name="dinov2"; embed each frame, mean-pool over frames. has_logits=False.

# ----------------------------------------------------------------------
# Metric helpers
# ----------------------------------------------------------------------
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

def kl_div(clean_logits, pert_logits):                # KL(clean || perturbed)
    sm = lambda x: np.exp(x - x.max()) / np.exp(x - x.max()).sum()
    p, q = sm(clean_logits), sm(pert_logits)
    return float((p * (np.log(p + 1e-12) - np.log(q + 1e-12))).sum())

# ----------------------------------------------------------------------
# Dataset indexing: parent-folder name == class label. Stratified train/val split.
# ----------------------------------------------------------------------
def index_videos(root, max_classes=None, per_class=None, val_frac=0.2, seed=0):
    rng = np.random.default_rng(seed)
    by_class = {}
    for p in sorted(glob.glob(f"{root}/**/*.mp4", recursive=True)):
        by_class.setdefault(os.path.basename(os.path.dirname(p)), []).append(p)
    classes = sorted(by_class)[:max_classes] if max_classes else sorted(by_class)
    cls2idx = {c: i for i, c in enumerate(classes)}
    train, val = [], []
    for c in classes:
        paths = by_class[c][:]
        rng.shuffle(paths)
        if per_class:
            paths = paths[:per_class]
        n_val = max(1, int(len(paths) * val_frac))
        val   += [(p, cls2idx[c]) for p in paths[:n_val]]
        train += [(p, cls2idx[c]) for p in paths[n_val:]]
    return train, val, classes

# ----------------------------------------------------------------------
# Metric (1): perturbation sensitivity. Decodes each video once.
# ----------------------------------------------------------------------
def perturbation_sensitivity(bk, items, max_videos=100):
    paths = [it[0] for it in items][:max_videos]
    names = [n for n in PERTURBATIONS if n != "clean"]
    cos, kl, kept = ({n: [] for n in names} for _ in range(3))
    for p in paths:
        frames = load_frames(p)
        base = bk.forward(clean(frames))
        for n in names:
            cur = bk.forward(PERTURBATIONS[n](frames))
            cos[n].append(cosine(base["emb"], cur["emb"]))
            if bk.has_logits:
                kl[n].append(kl_div(base["logits"], cur["logits"]))
                kept[n].append(int(base["logits"].argmax() == cur["logits"].argmax()))
    hdr = f"\n[{bk.name}] perturbation sensitivity  (n={len(paths)})\n" + f"{'pert':<12}{'emb cos':>10}"
    hdr += f"{'pred KL':>10}{'top1 kept':>11}" if bk.has_logits else ""
    print(hdr + "\n" + "-" * len(hdr.strip()))
    for n in names:
        row = f"{n:<12}{np.mean(cos[n]):>10.3f}"
        if bk.has_logits:
            row += f"{np.mean(kl[n]):>10.3f}{np.mean(kept[n]) * 100:>10.0f}%"
        print(row)

# ----------------------------------------------------------------------
# Metric (2): linear probe. Train on CLEAN, test on each perturbation.
# Decodes train once (clean) and val once (all perturbations).
# ----------------------------------------------------------------------
def linear_probe(bk, train_items, val_items):
    Xtr = np.stack([bk.forward(load_frames(p))["emb"] for p, _ in train_items])
    ytr = np.array([lab for _, lab in train_items])
    scaler = StandardScaler().fit(Xtr)
    probe = LogisticRegression(max_iter=2000).fit(scaler.transform(Xtr), ytr)

    yv = np.array([lab for _, lab in val_items])
    cols = {n: [] for n in PERTURBATIONS}
    for p, _ in val_items:                       # decode each val video once, embed under every perturbation
        frames = load_frames(p)
        for n, fn in PERTURBATIONS.items():
            cols[n].append(bk.forward(fn(frames))["emb"])

    print(f"\n[{bk.name}] linear-probe accuracy  (train={len(train_items)}, val={len(val_items)})")
    print(f"{'pert':<12}{'acc':>8}")
    print("-" * 20)
    for n in PERTURBATIONS:
        acc = probe.score(scaler.transform(np.stack(cols[n])), yv)
        print(f"{n:<12}{acc:>8.3f}")

# ----------------------------------------------------------------------
# Run
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Point at the attached/extracted dataset. index_videos assumes <class>/<video>.mp4 layout —
    # check the real structure first (e.g. !find /kaggle/input -name '*.mp4' | head).
    VIDEO_DIR = "/kaggle/input/datasets/rohanmallick/kinetics-train-5per/kinetics400_5per/kinetics400_5per/train"

    train_items, val_items, classes = index_videos(VIDEO_DIR, max_classes=20, per_class=20)
    print(f"{len(classes)} classes | {len(train_items)} train | {len(val_items)} val")

    # Lazy/one-at-a-time so both models never sit in VRAM together (T4 is 16GB).
    BACKBONES = {"videomae": VideoMAEBackbone, "dismo": DisMoBackbone}
    for key, Make in BACKBONES.items():
        bk = Make()
        perturbation_sensitivity(bk, val_items, max_videos=100)
        linear_probe(bk, train_items, val_items)
        del bk
        torch.cuda.empty_cache()

20 classes | 287 train | 67 val


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]


[videomae] perturbation sensitivity  (n=67)
pert           emb cos   pred KL  top1 kept
---------------------------------------------------------------------------------------
shuffle          0.948     0.266        84%
reverse          0.991     0.028        96%
freeze           0.865     0.685        67%
grayscale        0.941     0.306        82%
blur             0.983     0.061        93%

[videomae] linear-probe accuracy  (train=287, val=67)
pert             acc
--------------------
clean          0.985
shuffle        0.925
reverse        0.985
freeze         0.866
grayscale      0.881
blur           0.955


Using cache found in /root/.cache/torch/hub/CompVis_DisMo_main



[dismo] perturbation sensitivity  (n=67)
pert           emb cos
---------------------------------------------------------------
shuffle          0.719
reverse          0.752
freeze           0.145
grayscale        0.965
blur             0.971

[dismo] linear-probe accuracy  (train=287, val=67)
pert             acc
--------------------
clean          0.388
shuffle        0.239
reverse        0.328
freeze         0.045
grayscale      0.388
blur           0.299
